In [7]:
from datetime import datetime
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# Get environment variables
mdb_path = os.getenv("MDB")
lead_type_path = os.getenv("LeadType")

if not mdb_path or not os.path.exists(mdb_path):
    print(f"❌ MDB file path not found or not set in environment: {mdb_path}")
elif not lead_type_path or not os.path.exists(lead_type_path):
    print(f"❌ LeadType definition file path not found or not set in environment: {lead_type_path}")
else:
    print(f"📂 Loading MDB file: {mdb_path}")
    df_mdb = pd.read_excel(mdb_path)

    print(f"📂 Loading LeadType definition file: {lead_type_path}")
    df_lead_def = pd.read_excel(lead_type_path)

    # Required lookup columns from definition file
    lookup_cols = [
        "Sector",
        "Activity Description",
        "Proposal Status",
    ]

    # Validate columns exist in both dataframes
    missing_mdb = [col for col in lookup_cols if col not in df_mdb.columns]
    missing_def = [
        col for col in lookup_cols + ["FileType"] if col not in df_lead_def.columns
    ]

    if missing_mdb:
        print(f"❌ Missing columns in MDB file: {missing_mdb}")
    elif missing_def:
        print(f"❌ Missing columns in LeadType definition file: {missing_def}")
    else:
        # File types to process (excluding 'To be Removed')
        file_types = [
            "Non-Mineral_PotentialLeads",
            "Non-Mineral_ActiveLeads",
            "Mineral_PotentialLeads",
            "Mineral_ActiveLeads",
        ]

        # Drop duplicates in definition rules to prevent unintended row explosion during merge
        df_rules = df_lead_def[df_lead_def["FileType"].isin(file_types)][
            lookup_cols + ["FileType"]
        ].drop_duplicates()

        # Perform inner merge to find matching records based on the four criteria columns
        merged_df = pd.merge(
            df_mdb,
            df_rules,
            on=lookup_cols,
            how="inner",
        )

        # Columns to keep in the final generated files
        columns_to_keep = [
            "Proposal No.",
            "Year",
            "Location",
            "Proposal Status",
            "Project Name",
            "Project Proponent",
            "Sector",
            "Activity Description",
            "Form_No",
            "Date of Submission",
            "Comment",
        ]

        # Output directory for the new files (defaults to the same directory as MDB file if SILVER not set)
        output_dir = os.getenv("SILVERS")

        # Create a separate file for each FileType status, removing old ones if they exist
        for f_type in file_types:
            output_file_name = f"{f_type}.xlsx"
            output_file_path = os.path.join(output_dir, output_file_name)

            # Drop the old file if it already exists to ensure a fresh creation
            if os.path.exists(output_file_path):
                try:
                    os.remove(output_file_path)
                    print(f"🗑️ Removed existing file: {output_file_name}")
                except Exception as e:
                    print(f"⚠️ Could not remove existing file {output_file_name}: {e}")

            df_subset = merged_df[merged_df["FileType"] == f_type].copy()

            # Filter columns to keep only those present in the dataframe to avoid KeyErrors
            available_cols_to_keep = [col for col in columns_to_keep if col in df_subset.columns]
            df_subset = df_subset[available_cols_to_keep]

            df_subset.to_excel(output_file_path, index=False)
            print(
                f"✅ Created fresh file '{output_file_name}' with {len(df_subset)} records at: {output_file_path}"
            )

        print("\n🎉 All four lead files generated successfully!")

📂 Loading MDB file: F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - Silver\MasterDB.xlsx
📂 Loading LeadType definition file: F:\Chimney Work\Marketing\LeadGen\Data Architecture\4 - Control Lists\LeadType.xlsx
🗑️ Removed existing file: Non-Mineral_PotentialLeads.xlsx
✅ Created fresh file 'Non-Mineral_PotentialLeads.xlsx' with 3777 records at: F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - SILVER\Temp\Non-Mineral_PotentialLeads.xlsx
🗑️ Removed existing file: Non-Mineral_ActiveLeads.xlsx
✅ Created fresh file 'Non-Mineral_ActiveLeads.xlsx' with 3562 records at: F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - SILVER\Temp\Non-Mineral_ActiveLeads.xlsx
🗑️ Removed existing file: Mineral_PotentialLeads.xlsx
✅ Created fresh file 'Mineral_PotentialLeads.xlsx' with 11876 records at: F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - SILVER\Temp\Mineral_PotentialLeads.xlsx
🗑️ Removed existing file: Mineral_ActiveLeads.xlsx
✅ Created fresh file 'Mineral_ActiveLeads.xlsx'

In [8]:
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

def load_dataframe(file_path):
    """Loads CSV or Excel file into a DataFrame based on file extension."""
    if file_path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(file_path)
    return pd.read_csv(file_path)


def save_dataframe(df, file_path):
    """Saves DataFrame back to CSV or Excel based on file extension."""
    if file_path.lower().endswith((".xlsx", ".xls")):
        df.to_excel(file_path, index=False)
    else:
        df.to_csv(file_path, index=False)


def main():
    # 1. Fetch file paths from environment variables
    active_leads_path = os.environ.get("ActiveLeads")
    contact_path = os.environ.get("Contact")

    if not active_leads_path or not contact_path:
        raise ValueError(
            "Both 'ActiveLeads' and 'Contact' environment variables must be set."
        )

    # 2. Load the files
    df_active = load_dataframe(active_leads_path)
    df_contact = load_dataframe(contact_path)

    # 3. Define mapping key and requested columns to transfer
    key_col = "Proposal No."
    target_columns = [
        "proposal details",
        "Proposal URL",
        "Project Details XML",
        "Email_1",
        "Mobile_1",
        "Landline_1",
        "Email_2",
        "Landline_2",
        "Applicant_Name",
    ]

    # Ensure key exists in both DataFrames
    if key_col not in df_active.columns or key_col not in df_contact.columns:
        raise KeyError(
            f"The key column '{key_col}' must exist in both files."
        )

    # Filter available target columns present in Contact file
    available_cols = [
        col for col in target_columns if col in df_contact.columns
    ]

    # Drop target columns from ActiveLeads if they already exist (prevents duplicate suffix conflicts)
    cols_to_drop = [
        col for col in available_cols if col in df_active.columns
    ]
    if cols_to_drop:
        df_active = df_active.drop(columns=cols_to_drop)

    # 4. Prepare contact data (remove duplicates on join key to avoid row duplication)
    df_contact_subset = df_contact[[key_col] + available_cols].drop_duplicates(
        subset=[key_col]
    )

    # 5. Merge data (Left join keeps all original rows in ActiveLeads)
    df_updated = pd.merge(df_active, df_contact_subset, on=key_col, how="left")

    # 6. Save the updated file back to the ActiveLeads file path
    save_dataframe(df_updated, active_leads_path)
    print(
        f"Successfully added columns to {active_leads_path} mapped by '{key_col}'."
    )


if __name__ == "__main__":
    main()

Successfully added columns to F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - Silver\Temp\Non-Mineral_ActiveLeads.xlsx mapped by 'Proposal No.'.
